# Spondon: Stress Detection from a Low-Cost Multi-Sensor Wearable

Analysis code for *Spondon: A Low-Cost, Multi-Sensor Wearable for Understanding Stress in Resource-Constrained Countries* (ACM COMPASS 2026). [doi:10.1145/3811242.3819996](https://doi.org/10.1145/3811242.3819996)

Each participant completed a 5-minute **Baseline**, a 5-minute **Stroop Test**, and a **Recovery** period while wearing the Spondon device (MAX30102 PPG, GSR, body temperature). The participant data is not included in this repository; see `data/README.md` for the expected file format.


## 1. Setup and signal processing

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

# ---------------------------------------------------------
# CONFIGURATION: point DATA_DIR at the folder of participant CSV files
# ---------------------------------------------------------
DATA_DIR = 'data/'          # in Colab, e.g. '/content/drive/MyDrive/<folder>/'
FIG_DIR = 'figures/'
os.makedirs(FIG_DIR, exist_ok=True)

# ---------------------------------------------------------
# PAPER-READY GRAPHICS CONFIGURATION
# ---------------------------------------------------------
# This ensures all graphs generated have large, readable text and high resolution
plt.rcParams.update({
    'font.size': 20,              # Massive base font
    'axes.titlesize': 24,         # Massive title
    'axes.labelsize': 22,         # Massive X/Y labels
    'xtick.labelsize': 18,        # Big tick marks
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'legend.title_fontsize': 20,
    'figure.dpi': 300,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold'
})
sns.set_theme(style="whitegrid", palette="muted")

# ---------------------------------------------------------
# MASTER DATA PROCESSING FUNCTION
# ---------------------------------------------------------
def process_subject(file_path, subject_name):
    sensor_data, hrv_data = [], []
    header = ['Time', 'HR_Red', 'HR_IR', 'Inst_BPM', 'GSR_Raw', 'GSR_uS',
              'BodyTemp_C', 'AmbientTemp_C', 'ACC_X_G', 'ACC_Y_G', 'ACC_Z_G']

    with open(file_path, 'r') as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(',')]
            if "HRV_available" in line:
                metrics = {}
                for item in parts:
                    if ':' in item:
                        try:
                            key, val = item.split(':')
                            metrics[key.strip()] = float(val.strip()) if val.strip().lower() != 'true' else True
                        except: continue
                if metrics: hrv_data.append(metrics)
            else:
                clean_parts = parts[:11]
                if len(clean_parts) == 11 and ':' in clean_parts[0]:
                    sensor_data.append(clean_parts)

    # Create DataFrames
    df = pd.DataFrame(sensor_data, columns=header)
    df_hrv = pd.DataFrame(hrv_data)

    # Clean numeric columns
    num_cols = ['Inst_BPM', 'GSR_uS', 'BodyTemp_C']
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df['Inst_BPM'] = df['Inst_BPM'].replace(0, np.nan).interpolate()

    # Time and Phase alignment
    def to_secs(t_str):
        try:
            p = list(map(int, t_str.split(':')))
            return p[0]*3600 + p[1]*60 + p[2] if len(p)==3 else p[0]*60 + p[1]
        except: return None

    df['Elapsed_Sec'] = df['Time'].apply(to_secs)
    df['Elapsed_Sec'] -= df['Elapsed_Sec'].iloc[0]

    def get_phase(sec):
        if sec <= 300: return 'Baseline'
        elif sec <= 600: return 'Stroop Test'
        else: return 'Recovery'

    df['Phase'] = df['Elapsed_Sec'].apply(get_phase)

    # Smooth signals
    df['BPM_smooth'] = df['Inst_BPM'].rolling(10, center=True).mean()
    df['GSR_smooth'] = df['GSR_uS'].rolling(10, center=True).mean()
    df['Temp_smooth'] = df['BodyTemp_C'].rolling(10, center=True).mean()

    # Add subject labels
    df['Subject'] = subject_name
    df_hrv['Subject'] = subject_name

    # Align HRV phases
    if not df_hrv.empty:
        df_hrv['Minute'] = range(1, len(df_hrv) + 1)
        df_hrv['Phase'] = (df_hrv['Minute'] * 60).apply(get_phase)

    return df, df_hrv

print("Phase 1: Environment and Processing Engine Initialized.")

## 2. Load participant data (anonymized)

In [ ]:
# Optional: mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Find all participant CSV files
all_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

df_list = []
hrv_list = []

print(f"\nFound {len(all_files)} files. Beginning batch processing...\n")

# Each participant is given an anonymous ID (P01, P02, ...) so that
# no names or file names appear in outputs or figures.
for idx, file in enumerate(all_files, start=1):
    subject_id = f"P{idx:02d}"
    df, hrv = process_subject(file, subject_id)
    df_list.append(df)
    hrv_list.append(hrv)

if df_list:
    df_all = pd.concat(df_list, ignore_index=True)
    hrv_all = pd.concat(hrv_list, ignore_index=True)
    N_SUBJECTS = df_all['Subject'].nunique()
    print(f"All {N_SUBJECTS} participants merged.")
    print(f"Total continuous sensor rows: {len(df_all)}")
    print(f"Total HRV summary rows: {len(hrv_all)}")
    display(df_all.groupby('Subject')['Phase'].value_counts().unstack().fillna(0))
else:
    print("\nNo CSV files found. Check DATA_DIR.")


## 3. Population-level time series

In [ ]:
# =========================================================
# CELL 3: POPULATION-LEVEL TIME SERIES (MEAN & CONFIDENCE INTERVAL)
# =========================================================
print("Generating Population-Level Time Series...")

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# 1. Heart Rate (BPM)
sns.lineplot(data=df_all, x='Elapsed_Sec', y='BPM_smooth', color='red', ax=axes[0], linewidth=3)
axes[0].set_title(f"Average Heart Rate Response (N={N_SUBJECTS})", fontsize=24, fontweight='bold', pad=15)
axes[0].set_ylabel("Beats Per Minute\n(BPM)", fontsize=20, fontweight='bold')

# 2. Skin Conductance (GSR)
sns.lineplot(data=df_all, x='Elapsed_Sec', y='GSR_smooth', color='blue', ax=axes[1], linewidth=3)
axes[1].set_title(f"Average Skin Conductance (N={N_SUBJECTS})", fontsize=24, fontweight='bold', pad=15)
axes[1].set_ylabel("Skin Conductance\n(uS)", fontsize=20, fontweight='bold')

# 3. Body Temperature
sns.lineplot(data=df_all, x='Elapsed_Sec', y='Temp_smooth', color='orange', ax=axes[2], linewidth=3)
axes[2].set_title(f"Average Peripheral Body Temperature (N={N_SUBJECTS})", fontsize=24, fontweight='bold', pad=15)
axes[2].set_ylabel("Temperature\n(°C)", fontsize=20, fontweight='bold')

# Formatting the Backgrounds and Labels
max_sec = df_all['Elapsed_Sec'].max()

for ax in axes:
    # Add Shaded Phase Regions
    ax.axvspan(0, 300, color='green', alpha=0.08)
    ax.axvspan(300, 600, color='red', alpha=0.08)
    if max_sec > 600:
        ax.axvspan(600, max_sec, color='blue', alpha=0.08)

    ax.grid(True, alpha=0.4)

    ax.tick_params(axis='both', which='major', labelsize=18)

    # Add Text Labels inside the graph
    y_min, y_max = ax.get_ylim()
    y_pos = y_max - (y_max - y_min) * 0.1

    ax.text(150, y_pos, 'BASELINE', ha='center', fontsize=20, fontweight='bold', color='darkgreen', alpha=0.8)
    ax.text(450, y_pos, 'STROOP TEST', ha='center', fontsize=20, fontweight='bold', color='darkred', alpha=0.8)
    if max_sec > 600:
        center_rec = 600 + (max_sec - 600) / 2
        ax.text(center_rec, y_pos, 'RECOVERY', ha='center', fontsize=20, fontweight='bold', color='darkblue', alpha=0.8)

plt.xlabel("Elapsed Time (Seconds)", fontsize=22, fontweight='bold')
plt.tight_layout()

# Save the figure as a high-res PDF for Overleaf
plt.savefig(FIG_DIR + 'population_timeseries.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved figure: 'population_timeseries.pdf'")

## 4. Baseline-normalized skin conductance

In [ ]:
# =========================================================
# CELL 4: NORMALIZED GSR (PERCENTAGE CHANGE FROM BASELINE)
# =========================================================
print("Calculating Normalized GSR Percentages...")

# 1. Find the average resting Baseline GSR for EVERY individual subject
baseline_gsr = df_all[df_all['Phase'] == 'Baseline'].groupby('Subject')['GSR_smooth'].mean()

# 2. Map this personal baseline back to the main dataset
df_all['Personal_Baseline_GSR'] = df_all['Subject'].map(baseline_gsr)

# 3. Calculate Percentage Change: ((Current - Baseline) / Baseline) * 100
df_all['GSR_Normalized_Pct'] = ((df_all['GSR_smooth'] - df_all['Personal_Baseline_GSR']) / df_all['Personal_Baseline_GSR']) * 100

# 4. Plot the Normalized Data
plt.figure(figsize=(16, 7))

# Plot the population average of the PERCENTAGE increase
sns.lineplot(data=df_all, x='Elapsed_Sec', y='GSR_Normalized_Pct', color='purple', linewidth=4)

# Add reference lines and shading
plt.axhline(0, color='black', linestyle='--', linewidth=3, label='0% (Resting Baseline)')
plt.axvspan(300, 600, color='red', alpha=0.1, label='Stroop Test (Stress Phase)')

plt.title(f"Normalized Sympathetic Arousal Across N={N_SUBJECTS} Subjects", fontsize=24, fontweight='bold', pad=20)
plt.ylabel("Increase in Skin Conductance (%)", fontsize=22, fontweight='bold')
plt.xlabel("Elapsed Time (Seconds)", fontsize=22, fontweight='bold')

plt.xticks(fontsize=18, fontweight='bold')
plt.yticks(fontsize=18, fontweight='bold')

# Make the legend big and solid
plt.legend(loc='upper right', fontsize=18, framealpha=1.0)
plt.grid(True, alpha=0.4)
plt.tight_layout()

# Save for publication
plt.savefig(FIG_DIR + 'normalized_gsr.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved figure: 'normalized_gsr.pdf'")

## 5. Heart rate variability (RMSSD) by phase

In [ ]:
# =========================================================
# CELL 5: HRV (RMSSD) STATISTICAL PROOF
# =========================================================
print("Generating HRV Autonomic Response Analysis...")

# 1. Filter out any missing HRV rows
hrv_clean = hrv_all.dropna(subset=['RMSSD']).copy()

# 2. Set the order of the phases
phase_order = ['Baseline', 'Stroop Test', 'Recovery']

# 3. Create the Paper-Ready Plot
plt.figure(figsize=(10, 8)) # Made slightly taller for the bigger title

# We use a barplot with built-in Confidence Interval (error bars)
ax = sns.barplot(
    data=hrv_clean,
    x='Phase',
    y='RMSSD',
    order=phase_order,
    palette=['#2ecc71', '#e74c3c', '#3498db'],
    capsize=0.1,      # Adds the little caps to the error bars
    err_kws={'linewidth': 3, 'color': 'black'} # Thicker error bars for visibility
)

plt.title(f"Population HRV Response to Cognitive Load (N={N_SUBJECTS})", fontsize=24, fontweight='bold', pad=20)
plt.ylabel("RMSSD (ms)\n[Lower = Higher Stress]", fontsize=20, fontweight='bold') # Added a line break just in case!
plt.xlabel("Experimental Phase", fontsize=22, fontweight='bold')

plt.xticks(fontsize=20, fontweight='bold')
plt.yticks(fontsize=18, fontweight='bold')

for i, phase in enumerate(phase_order):
    mean_val = hrv_clean[hrv_clean['Phase'] == phase]['RMSSD'].mean()
    ax.text(i, mean_val / 2, f"{mean_val:.1f} ms", ha='center', color='white',
            fontweight='bold', fontsize=24, bbox=dict(facecolor='black', alpha=0.5, edgecolor='none'))

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

# Save for publication
plt.savefig(FIG_DIR + 'hrv_population_stats.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved figure: 'hrv_population_stats.pdf'")

## 6. LOSO classification with raw features

In [ ]:
# =========================================================
# CELL 6: LEAVE-ONE-SUBJECT-OUT (LOSO) MACHINE LEARNING & RESULTS
# =========================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore') # Hides annoying sci-kit learn warnings

print("Initiating Leave-One-Subject-Out (LOSO) Cross-Validation...\n")

# 1. Prepare ML Data
# Only keep Baseline (0) and Stroop Test (1)
ml_df = df_all[df_all['Phase'].isin(['Baseline', 'Stroop Test'])].copy()

# Features to use
features = ['BPM_smooth', 'GSR_smooth', 'Temp_smooth']
ml_df = ml_df.dropna(subset=features)
ml_df['Label'] = ml_df['Phase'].map({'Baseline': 0, 'Stroop Test': 1})

# Get list of unique subjects
subjects = ml_df['Subject'].unique()
loso_accuracies = []

print(f"Training on N={len(subjects)} subjects. This will take a few seconds...\n")

# 2. The LOSO Loop
for test_subject in subjects:
    # Split data: 1 person for testing, the rest for training
    train_data = ml_df[ml_df['Subject'] != test_subject]
    test_data = ml_df[ml_df['Subject'] == test_subject]

    X_train, y_train = train_data[features], train_data['Label']
    X_test, y_test = test_data[features], test_data['Label']

    # Train the Random Forest
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf_model.fit(X_train, y_train)

    # Test the model
    y_pred = rf_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    loso_accuracies.append(acc)

    print(f"Tested on {test_subject} -> Accuracy: {acc * 100:.1f}%")

# 3. Calculate Overall Performance
avg_loso_acc = np.mean(loso_accuracies) * 100
print(f"\nAverage LOSO Accuracy: {avg_loso_acc:.2f}%")

# Population HRV (RMSSD) change from Baseline to Stroop
mean_baseline_rmssd = hrv_clean[hrv_clean['Phase']=='Baseline']['RMSSD'].mean()
mean_stroop_rmssd = hrv_clean[hrv_clean['Phase']=='Stroop Test']['RMSSD'].mean()
hrv_drop_pct = ((mean_baseline_rmssd - mean_stroop_rmssd) / mean_baseline_rmssd) * 100
print(f"Population HRV (RMSSD) drop during Stroop: {hrv_drop_pct:.1f}%")


## 7. LOSO classification with baseline-normalized features

In [ ]:
# =========================================================
# CELL 7: ADVANCED LOSO ML (NORMALIZED FEATURES)
# =========================================================
print("Initiating Advanced ML with Normalized Features...\n")

# 1. We need to normalize BPM and Temp the same way we normalized GSR in Phase 3
# Find baselines
baseline_bpm = df_all[df_all['Phase'] == 'Baseline'].groupby('Subject')['BPM_smooth'].mean()
baseline_temp = df_all[df_all['Phase'] == 'Baseline'].groupby('Subject')['Temp_smooth'].mean()

# Map baselines to the main dataset
df_all['Personal_Baseline_BPM'] = df_all['Subject'].map(baseline_bpm)
df_all['Personal_Baseline_Temp'] = df_all['Subject'].map(baseline_temp)

# Calculate Percentage Changes
df_all['BPM_Norm_Pct'] = ((df_all['BPM_smooth'] - df_all['Personal_Baseline_BPM']) / df_all['Personal_Baseline_BPM']) * 100
df_all['Temp_Norm_Pct'] = ((df_all['Temp_smooth'] - df_all['Personal_Baseline_Temp']) / df_all['Personal_Baseline_Temp']) * 100

# 2. Prepare ML Data using our new NORMALIZED columns
ml_df_norm = df_all[df_all['Phase'].isin(['Baseline', 'Stroop Test'])].copy()

# USE THE PERCENTAGE COLUMNS INSTEAD OF RAW COLUMNS
norm_features = ['BPM_Norm_Pct', 'GSR_Normalized_Pct', 'Temp_Norm_Pct']
ml_df_norm = ml_df_norm.dropna(subset=norm_features)
ml_df_norm['Label'] = ml_df_norm['Phase'].map({'Baseline': 0, 'Stroop Test': 1})

subjects = ml_df_norm['Subject'].unique()
loso_accuracies_norm = []

print(f"Training Normalized Model on N={N_SUBJECTS} subjects...\n")

for test_subject in subjects:
    train_data = ml_df_norm[ml_df_norm['Subject'] != test_subject]
    test_data = ml_df_norm[ml_df_norm['Subject'] == test_subject]

    X_train, y_train = train_data[norm_features], train_data['Label']
    X_test, y_test = test_data[norm_features], test_data['Label']

    rf_model_norm = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf_model_norm.fit(X_train, y_train)

    y_pred = rf_model_norm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    loso_accuracies_norm.append(acc)

    print(f"Tested on {test_subject} -> Accuracy: {acc * 100:.1f}%")

avg_loso_acc_norm = np.mean(loso_accuracies_norm) * 100
print(f"\nAverage LOSO Accuracy (Normalized): {avg_loso_acc_norm:.2f}%")
print(f"Improvement over raw data: {avg_loso_acc_norm - avg_loso_acc:.2f} percentage points")

## 8. Feature importance

In [ ]:
# =========================================================
# CELL 8: FINAL DASHBOARD & FEATURE IMPORTANCE
# =========================================================
print("Plotting feature importance...\n")

fig, axes = plt.subplots(1, 1, figsize=(10, 5))

# 1. Feature Importance Graph
# Importance from the model of the final LOSO fold (indicative only)
importances = rf_model_norm.feature_importances_
sns.barplot(x=importances, y=norm_features, palette="magma", ax=axes)

axes.set_title("Machine Learning Feature Importance (Normalized)", fontsize=16, fontweight='bold')
axes.set_xlabel("Impact on Stress Prediction (0 to 1)", fontsize=14, fontweight='bold')
axes.set_yticklabels(['Heart Rate (% Change)', 'Skin Conductance (% Change)', 'Body Temp (% Change)'], fontweight='bold')
plt.tight_layout()

# Save the feature importance graph
plt.savefig(FIG_DIR + 'ml_feature_importance.pdf', format='pdf', bbox_inches='tight')
plt.show()



## 9. Model comparison (LOSO)

In [ ]:
# =========================================================
# CELL 9: THE ALGORITHM SHOWDOWN (MODEL COMPARISON)
# =========================================================
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print("Initiating Algorithm Showdown (LOSO Cross-Validation)...\n")
print(f"Training 5 different Machine Learning models on N={N_SUBJECTS} subjects. This might take a minute...\n")

# 1. Define the models we want to compare
models = {
    'Random Forest': RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42),
    'SVM (Linear)': SVC(kernel='linear', random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# Dictionary to store the results
results = {model_name: [] for model_name in models.keys()}

# 2. Get subjects list from our normalized dataset (from Cell 7)
subjects = ml_df_norm['Subject'].unique()

# 3. The Grand LOSO Loop
for test_subject in subjects:
    # Split data: 1 participant for testing, the rest for training
    train_data = ml_df_norm[ml_df_norm['Subject'] != test_subject]
    test_data = ml_df_norm[ml_df_norm['Subject'] == test_subject]

    X_train, y_train = train_data[norm_features], train_data['Label']
    X_test, y_test = test_data[norm_features], test_data['Label']

    # Train and test EVERY model on this specific split
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results[model_name].append(acc)

# 4. Calculate overall average accuracies
avg_accuracies = {model_name: np.mean(acc_list) * 100 for model_name, acc_list in results.items()}

# Print Text Results in order of performance
print("Final algorithm accuracies:")
for model_name, acc in sorted(avg_accuracies.items(), key=lambda x: x[1], reverse=True):
    print(f"{model_name}: {acc:.2f}%")

# =========================================================
# 5. VISUALIZATION: The Showdown Bar Chart
# =========================================================
plt.figure(figsize=(12, 8))

# Convert to dataframe for easy Seaborn plotting and sort from highest to lowest
results_df = pd.DataFrame(list(avg_accuracies.items()), columns=['Algorithm', 'Accuracy (%)'])
results_df = results_df.sort_values(by='Accuracy (%)', ascending=False)

# Create the barplot
ax = sns.barplot(data=results_df, x='Algorithm', y='Accuracy (%)', palette='viridis')

# Massive explicitly defined fonts for the numbers on top of the bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=5, fontweight='bold', fontsize=22)

plt.title("Algorithm Showdown: Stress Prediction using Normalized Sensors", fontsize=24, fontweight='bold', pad=40)

plt.ylabel("LOSO Cross-Validation\nAccuracy (%)", fontsize=22, fontweight='bold')

plt.xlabel("Machine Learning Algorithm", fontsize=22, fontweight='bold')
plt.ylim(0, 100) # Lock the Y-axis from 0 to 100%
plt.grid(axis='y', alpha=0.3)

# Massive tick labels AND a 45-degree rotation for the long model names
plt.xticks(rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.yticks(fontsize=20, fontweight='bold')

plt.tight_layout()

# Save for publication
plt.savefig(FIG_DIR + 'algorithm_showdown.pdf', format='pdf', bbox_inches='tight')
plt.show()

print("\nSaved figure: 'algorithm_showdown.pdf'")

## 10. Rolling-window (10 s) features

In [ ]:
# =========================================================
# CELL 10: ADVANCED ML - ROLLING WINDOW FEATURES (EPOCHS)
# =========================================================
print("Initiating Advanced Feature Engineering (10-Second Windows)...\n")

# 1. Create a copy of our normalized dataframe
epoch_df = ml_df_norm.copy()

# 2. Calculate Rolling Standard Deviation (How much is the signal wobbling?)
# We group by subject so we don't accidentally mix two different people's data
epoch_df['BPM_Std_10s'] = epoch_df.groupby('Subject')['BPM_Norm_Pct'].rolling(10, min_periods=1).std().reset_index(level=0, drop=True)
epoch_df['GSR_Std_10s'] = epoch_df.groupby('Subject')['GSR_Normalized_Pct'].rolling(10, min_periods=1).std().reset_index(level=0, drop=True)

# 3. Calculate the "Slope" (Difference between current value and the value 10 seconds ago)
# This tells the model if the sweat is actively RISING or FALLING
epoch_df['GSR_Slope_10s'] = epoch_df.groupby('Subject')['GSR_Normalized_Pct'].diff(10)
epoch_df['BPM_Slope_10s'] = epoch_df.groupby('Subject')['BPM_Norm_Pct'].diff(10)

# 4. Drop the rows where we couldn't calculate a 10-second history (the first 10 seconds of the experiment)
epoch_features = ['BPM_Norm_Pct', 'GSR_Normalized_Pct', 'Temp_Norm_Pct',
                  'BPM_Std_10s', 'GSR_Std_10s', 'GSR_Slope_10s', 'BPM_Slope_10s']
epoch_df = epoch_df.dropna(subset=epoch_features)

print(f"Training Random Forest with new TIME-BASED features on N={N_SUBJECTS} subjects...\n")

loso_accuracies_epoch = []
subjects = epoch_df['Subject'].unique()

# 5. The LOSO Loop
for test_subject in subjects:
    train_data = epoch_df[epoch_df['Subject'] != test_subject]
    test_data = epoch_df[epoch_df['Subject'] == test_subject]

    X_train, y_train = train_data[epoch_features], train_data['Label']
    X_test, y_test = test_data[epoch_features], test_data['Label']

    # We use Random Forest since it won the showdown!
    rf_model_epoch = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf_model_epoch.fit(X_train, y_train)

    y_pred = rf_model_epoch.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    loso_accuracies_epoch.append(acc)

    print(f"Tested on {test_subject} -> Accuracy: {acc * 100:.1f}%")

avg_loso_acc_epoch = np.mean(loso_accuracies_epoch) * 100
print(f"\nFinal epoch LOSO ACCURACY: {avg_loso_acc_epoch:.2f}%")

# Plot the new Feature Importance
plt.figure(figsize=(10, 5))
importances_epoch = rf_model_epoch.feature_importances_
sns.barplot(x=importances_epoch, y=epoch_features, palette="mako")
plt.title("Feature Importance: Time-Based Windowing", fontsize=16, fontweight='bold')
plt.xlabel("Impact on Stress Prediction", fontsize=14)
plt.tight_layout()
plt.show()

## 11. Three-class classification: Baseline vs. Stroop vs. Recovery

In [ ]:
# =========================================================
# CELL 11: MULTI-CLASS ML (BASELINE vs STRESS vs RECOVERY)
# =========================================================
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("Initiating 3-Class ML (Baseline vs Stroop vs Recovery)...\n")

# 1. Prepare the Multi-Class Dataset using our whole normalized dataset
multi_df = df_all.copy()

# Re-calculate the powerful 10-second window features for ALL phases
multi_df['BPM_Std_10s'] = multi_df.groupby('Subject')['BPM_Norm_Pct'].rolling(10, min_periods=1).std().reset_index(level=0, drop=True)
multi_df['GSR_Std_10s'] = multi_df.groupby('Subject')['GSR_Normalized_Pct'].rolling(10, min_periods=1).std().reset_index(level=0, drop=True)
multi_df['GSR_Slope_10s'] = multi_df.groupby('Subject')['GSR_Normalized_Pct'].diff(10)
multi_df['BPM_Slope_10s'] = multi_df.groupby('Subject')['BPM_Norm_Pct'].diff(10)

# 2. Filter nans and map all THREE labels
multi_features = ['BPM_Norm_Pct', 'GSR_Normalized_Pct', 'Temp_Norm_Pct',
                  'BPM_Std_10s', 'GSR_Std_10s', 'GSR_Slope_10s', 'BPM_Slope_10s']
multi_df = multi_df.dropna(subset=multi_features)

# Label Mapping: 0=Relaxed, 1=Stressed, 2=Recovering
multi_df['Label'] = multi_df['Phase'].map({'Baseline': 0, 'Stroop Test': 1, 'Recovery': 2})
multi_df = multi_df.dropna(subset=['Label']) # Just in case any weird phase names exist

subjects = multi_df['Subject'].unique()
all_y_true = []
all_y_pred = []

print(f"Training 3-Class Random Forest Model on N={N_SUBJECTS} subjects...\n")

# 3. The LOSO Loop
for test_subject in subjects:
    train_data = multi_df[multi_df['Subject'] != test_subject]
    test_data = multi_df[multi_df['Subject'] == test_subject]

    X_train, y_train = train_data[multi_features], train_data['Label']
    X_test, y_test = test_data[multi_features], test_data['Label']

    rf_multi = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
    rf_multi.fit(X_train, y_train)

    y_pred = rf_multi.predict(X_test)

    # Store predictions for the giant confusion matrix
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)

# Calculate overall 3-class accuracy
multi_acc = accuracy_score(all_y_true, all_y_pred) * 100
print(f"Final 3-class LOSO ACCURACY: {multi_acc:.2f}% (chance level: 33%)\n")

# =========================================================
# 4. VISUALIZATION: The Confusion Matrix Heatmap
# =========================================================
cm = confusion_matrix(all_y_true, all_y_pred)
# Normalize the matrix to show percentages instead of raw thousands of seconds
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(10, 8))

ax = sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues',
                 xticklabels=['Baseline', 'Stroop Test', 'Recovery'],
                 yticklabels=['Baseline', 'Stroop Test', 'Recovery'],
                 annot_kws={"size": 24, "weight": "bold"})

plt.title("Multi-Class Stress Classification", fontsize=24, fontweight='bold', pad=20)
plt.ylabel('True Experimental Phase', fontsize=22, fontweight='bold')
plt.xlabel('Algorithm Prediction', fontsize=22, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.yticks(rotation=0, fontsize=20, fontweight='bold')

# Adjust text inside heatmap for readability (adds the % sign)
for t in ax.texts: t.set_text(t.get_text() + " %")

plt.tight_layout()

# Save for publication
plt.savefig(FIG_DIR + 'multiclass_confusion_matrix.pdf', format='pdf', bbox_inches='tight')
plt.show()
print("Saved figure: 'multiclass_confusion_matrix.pdf'")

## 12. Results table (LaTeX)

In [ ]:
# =========================================================
# CELL 12: GENERATE RESULTS TABLE
# =========================================================
print("Generating numerical findings table...\n")

# Gather the metrics calculated in previous cells
# (We use a try-except block just in case a variable was missed)
try:
    drop_hrv = f"{hrv_drop_pct:.1f}\\%"
except: drop_hrv = "N/A"

try:
    acc_raw = f"{avg_loso_acc:.1f}\\%"
except: acc_raw = "N/A"

try:
    acc_norm = f"{avg_loso_acc_norm:.1f}\\%"
except: acc_norm = "N/A"

try:
    acc_epoch = f"{avg_loso_acc_epoch:.1f}\\%"
except: acc_epoch = "N/A"

try:
    acc_multi = f"{multi_acc:.1f}\\%"
except: acc_multi = "N/A"

# Generate the LaTeX Code for Overleaf
latex_code = f"""
\\begin{{table}}[htbp]
\\centering
\\caption{{Summary of Physiological and Machine Learning Findings (N={N_SUBJECTS})}}
\\label{{tab:numerical_findings}}
\\begin{{tabular}}{{l l c}}
\\toprule
\\textbf{{Analysis Category}} & \\textbf{{Specific Metric}} & \\textbf{{Result}} \\\\
\\midrule
\\textit{{Autonomic Response}} & Population HRV (RMSSD) Drop & {drop_hrv} \\\\
\\midrule
\\textit{{Binary Classification}} & LOSO Accuracy (Raw Sensors) & {acc_raw} \\\\
 & LOSO Accuracy (Normalized Sensors) & {acc_norm} \\\\
 & LOSO Accuracy (10s Epoch Features) & {acc_epoch} \\\\
\\midrule
\\textit{{Multi-Class Classification}}& 3-Class Accuracy (Baseline/Stroop/Recovery) & {acc_multi} \\\\
\\bottomrule
\\end{{tabular}}
\\vspace{{1ex}}
\\\\ \\small{{\\textit{{Note:}} LOSO = Leave-One-Subject-Out Cross Validation. 3-Class random baseline is 33\\%.}}
\\end{{table}}
"""

print("LaTeX table:\n")
print(latex_code)

## 13. Inferential statistics (Wilcoxon signed-rank)

In [ ]:
import scipy.stats as stats
import pandas as pd

print("CALCULATING INFERENTIAL STATISTICS (WILCOXON TEST)...\n")

# 1. HRV (RMSSD) Statistic
# Get each subject's Baseline and Stroop RMSSD
hrv_base = hrv_clean[hrv_clean['Phase'] == 'Baseline'].groupby('Subject')['RMSSD'].mean()
hrv_stress = hrv_clean[hrv_clean['Phase'] == 'Stroop Test'].groupby('Subject')['RMSSD'].mean()

# Align the data to ensure we are comparing the exact same subjects
aligned_hrv = pd.DataFrame({'Base': hrv_base, 'Stress': hrv_stress}).dropna()

# Run the paired Wilcoxon test
stat_hrv, p_hrv = stats.wilcoxon(aligned_hrv['Base'], aligned_hrv['Stress'])
print(f"HRV (RMSSD) Wilcoxon Test: p-value = {p_hrv:.4f}")
print("HRV change is " + ("statistically significant" if p_hrv < 0.05 else "not statistically significant") + " at alpha = 0.05\n")

# 2. GSR Statistic
# Get each subject's Baseline and Stroop Mean GSR
gsr_base = df_all[df_all['Phase'] == 'Baseline'].groupby('Subject')['GSR_smooth'].mean()
gsr_stress = df_all[df_all['Phase'] == 'Stroop Test'].groupby('Subject')['GSR_smooth'].mean()

aligned_gsr = pd.DataFrame({'Base': gsr_base, 'Stress': gsr_stress}).dropna()

# Run the paired Wilcoxon test
stat_gsr, p_gsr = stats.wilcoxon(aligned_gsr['Base'], aligned_gsr['Stress'])
print(f"GSR Wilcoxon Test: p-value = {p_gsr:.4f}")
print("GSR change is " + ("statistically significant" if p_gsr < 0.05 else "not statistically significant") + " at alpha = 0.05")